# 10 · Baseline equivalence check: our scores vs the official code (design doc §8.8.15)

**How to run**: pick a Colab server with an **A100 GPU** and click Run All; about 10 minutes. Results go to `DATA_DIR/10_official_equivalence/`.

**What it computes**: 80 texts (20 from each of the three generators, 20 human) are scored twice, (a) with this study's batched scorer (as in 07) and (b) with the official code copied verbatim, to check that both agree. Official code: Fast-DetectGPT `scripts/fast_detect_gpt.py` and `scripts/baselines.py` (baoguangsheng/fast-detect-gpt, commit 971b0520, 2026-02-07); Binoculars `binoculars/metrics.py` (ahans30/Binoculars, commit c8ae2f90, 2024-05-14). Models as in 07: scorer/performer Falcon-7B-Instruct, reference/observer Falcon-7B.


In [ ]:
# 1. GPU and Google Drive
import torch, os
assert torch.cuda.is_available(), "No GPU"
free = torch.cuda.mem_get_info()[0] / 2**30
assert free > 35, f"Only {free:.0f} GB of GPU memory is free: another notebook is using this GPU; terminate the other sessions first."
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
OUT = "10_official_equivalence"; os.makedirs(OUT, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"


In [ ]:
# 2. Packages and sample: 20 continuation parts (without prefix) per generator from 06_scaled_sweep and 20 human continuations
%pip install -q -U transformers huggingface-hub accelerate
import json, time, sys, platform, hashlib
from datetime import datetime, timezone
import numpy as np, pandas as pd, transformers
rng = np.random.default_rng(20260926)
samples = []
for f, n in (("06_scaled_sweep/gen_Qwen2.5-3B_T0.8.json", 20), ("06_scaled_sweep/gen_phi-2_T1.0.json", 20), ("06_scaled_sweep/gen_SmolLM2-1.7B_T0.94.json", 20)):
    rows = json.load(open(f)); pick = rng.choice(len(rows), n, replace=False)
    samples += [dict(src=f, id=rows[i]["id"], text=rows[i]["ai"]) for i in pick]
hum = json.load(open("06_scaled_sweep/humans_Qwen2.5-3B.json")); pick = rng.choice(len(hum), 20, replace=False)
samples += [dict(src="humans", id=hum[i]["id"], text=hum[i]["human"]) for i in pick]
print(len(samples), "texts; text hash", hashlib.sha256("".join(s["text"] for s in samples).encode()).hexdigest()[:12])


In [ ]:
# 3. Load the two Falcon models
from transformers import AutoTokenizer, AutoModelForCausalLM
SCORER, OBSERVER, MAX_TOK, BS = "tiiuae/falcon-7b-instruct", "tiiuae/falcon-7b", 512, 8
for _n in ("p_model", "q_model"):
    globals().pop(_n, None)
import gc; gc.collect(); torch.cuda.empty_cache()
tok = AutoTokenizer.from_pretrained(SCORER); tok.padding_side = "right"
pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
p_model = AutoModelForCausalLM.from_pretrained(SCORER, dtype=torch.bfloat16).cuda().eval()   # scorer / performer
q_model = AutoModelForCausalLM.from_pretrained(OBSERVER, dtype=torch.bfloat16).cuda().eval() # reference / observer
print("loaded", f"{torch.cuda.memory_allocated() / 2**30:.1f} GB")


In [ ]:
# 4. (a) This study's batched scoring: the same 12 rows as 07_falcon_dual_scoring.ipynb, here over whole texts (k = 1, no prefix cut), then four scores
@torch.no_grad()
def ours(texts):
    enc = [tok(t, truncation=True, max_length=MAX_TOK)["input_ids"] for t in texts]
    out = []
    for s in range(0, len(enc), BS):
        chunk = enc[s:s + BS]; L = max(len(e) for e in chunk)
        x = torch.full((len(chunk), L), pad); att = torch.zeros((len(chunk), L), dtype=torch.long)
        for i, ids in enumerate(chunk):
            x[i, :len(ids)] = torch.tensor(ids); att[i, :len(ids)] = 1
        x, att = x.cuda(), att.cuda()
        lg_p = p_model(input_ids=x, attention_mask=att).logits; lg_q = q_model(input_ids=x, attention_mask=att).logits
        for i, ids in enumerate(chunk):
            n = len(ids)
            lp = torch.log_softmax(lg_p[i, :n - 1].float(), -1); lq = torch.log_softmax(lg_q[i, :n - 1].float(), -1)
            tgt = torch.tensor(ids[1:], device=lp.device); q = lq.exp()
            s_p = -lp.gather(1, tgt[:, None])[:, 0]; X = (q * lp).sum(-1); XV = (q * lp * lp).sum(-1) - X ** 2
            rank = (lp > lp.gather(1, tgt[:, None])).sum(-1).float()
            out.append(dict(ours_fdg=float((-s_p.sum() - X.sum()) / XV.sum().sqrt()), ours_bino=float(s_p.mean() / (-X.mean())),
                            ours_loglik=float(-s_p.mean()), ours_logrank=float(-torch.log(rank + 1).mean())))
    return out

# 4. (b) Official code, copied verbatim
def get_sampling_discrepancy_analytic(logits_ref, logits_score, labels):   # fast-detect-gpt scripts/fast_detect_gpt.py @971b0520
    assert logits_ref.shape[0] == 1
    assert logits_score.shape[0] == 1
    assert labels.shape[0] == 1
    if logits_ref.size(-1) != logits_score.size(-1):
        vocab_size = min(logits_ref.size(-1), logits_score.size(-1))
        logits_ref = logits_ref[:, :, :vocab_size]
        logits_score = logits_score[:, :, :vocab_size]
    labels = labels.unsqueeze(-1) if labels.ndim == logits_score.ndim - 1 else labels
    lprobs_score = torch.log_softmax(logits_score, dim=-1)
    probs_ref = torch.softmax(logits_ref, dim=-1)
    log_likelihood = lprobs_score.gather(dim=-1, index=labels).squeeze(-1)
    mean_ref = (probs_ref * lprobs_score).sum(dim=-1)
    var_ref = (probs_ref * torch.square(lprobs_score)).sum(dim=-1) - torch.square(mean_ref)
    discrepancy = (log_likelihood.sum(dim=-1) - mean_ref.sum(dim=-1)) / var_ref.sum(dim=-1).sqrt()
    discrepancy = discrepancy.mean()
    return discrepancy.item()

def get_likelihood(logits, labels):   # fast-detect-gpt scripts/baselines.py @971b0520
    assert logits.shape[0] == 1
    assert labels.shape[0] == 1
    logits = logits.view(-1, logits.shape[-1])
    labels = labels.view(-1)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    log_likelihood = log_probs.gather(dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
    return log_likelihood.mean().item()

def get_logrank(logits, labels):   # fast-detect-gpt scripts/baselines.py @971b0520
    assert logits.shape[0] == 1
    assert labels.shape[0] == 1
    matches = (logits.argsort(-1, descending=True) == labels.unsqueeze(-1)).nonzero()
    assert matches.shape[1] == 3, f"Expected 3 dimensions in matches tensor, got {matches.shape}"
    ranks, timesteps = matches[:, -1], matches[:, -2]
    assert (timesteps == torch.arange(len(timesteps)).to(timesteps.device)).all(), "Expected one match per timestep"
    ranks = ranks.float() + 1
    ranks = torch.log(ranks)
    return -ranks.mean().item()

ce_loss_fn = torch.nn.CrossEntropyLoss(reduction="none"); softmax_fn = torch.nn.Softmax(dim=-1)
def perplexity(encoding, logits, median=False, temperature=1.0):   # Binoculars binoculars/metrics.py @c8ae2f90
    shifted_logits = logits[..., :-1, :].contiguous() / temperature
    shifted_labels = encoding.input_ids[..., 1:].contiguous()
    shifted_attention_mask = encoding.attention_mask[..., 1:].contiguous()
    ppl = (ce_loss_fn(shifted_logits.transpose(1, 2), shifted_labels) * shifted_attention_mask).sum(1) / shifted_attention_mask.sum(1)
    return ppl.to("cpu").float().numpy()

def entropy(p_logits, q_logits, encoding, pad_token_id, median=False, sample_p=False, temperature=1.0):   # Binoculars @c8ae2f90
    vocab_size = p_logits.shape[-1]
    total_tokens_available = q_logits.shape[-2]
    p_scores, q_scores = p_logits / temperature, q_logits / temperature
    p_proba = softmax_fn(p_scores).view(-1, vocab_size)
    q_scores = q_scores.view(-1, vocab_size)
    ce = ce_loss_fn(input=q_scores, target=p_proba).view(-1, total_tokens_available)
    padding_mask = (encoding.input_ids != pad_token_id).type(torch.uint8)
    agg_ce = (((ce * padding_mask).sum(1) / padding_mask.sum(1)).to("cpu").float().numpy())
    return agg_ce

@torch.no_grad()
def official(text):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_TOK, return_token_type_ids=False).to("cuda")
    lg_p = p_model(**enc).logits.float(); lg_q = q_model(**enc).logits.float()
    labels = enc.input_ids[:, 1:]
    fdg = get_sampling_discrepancy_analytic(lg_q[:, :-1], lg_p[:, :-1], labels)             # ref = observer(falcon-7b), score = falcon-7b-instruct
    ppl = perplexity(enc, lg_p); x_ppl = entropy(lg_q, lg_p, enc, pad)   # the official detector.py passes the full logits (last position included) to entropy with the full input_ids as mask; dropping the last position breaks the lengths
    return dict(off_fdg=fdg, off_bino=float(ppl[0] / x_ppl[0]), off_loglik=get_likelihood(lg_p[:, :-1], labels), off_logrank=get_logrank(lg_p[:, :-1], labels))

texts = [s["text"] for s in samples]
A = pd.DataFrame(ours(texts)); B = pd.DataFrame([official(t) for t in texts])
R = pd.concat([pd.DataFrame(samples).drop(columns="text"), A, B], axis=1)
for m in ("fdg", "bino", "loglik", "logrank"):
    d = (R[f"ours_{m}"] - R[f"off_{m}"]).abs()
    print(f"{m:8s} max abs diff {d.max():.2e}  mean {d.mean():.2e}  Pearson {np.corrcoef(R[f'ours_{m}'], R[f'off_{m}'])[0, 1]:.6f}")
R.to_csv(f"{OUT}/equivalence_check.csv", index=False)


In [ ]:
# 5. The official Binoculars entropy() masks with input_ids != pad; if Falcon's pad id equals a real token, one position is lost. Check that no sample contains it, and record the environment.
ids_all = [tok(t, truncation=True, max_length=MAX_TOK)["input_ids"] for t in texts]
print("Texts containing the pad id:", sum(pad in ids for ids in ids_all))
info = dict(completed_at_utc=datetime.now(timezone.utc).isoformat(), n_texts=len(texts), scorer=SCORER, observer=OBSERVER, max_tokens=MAX_TOK,
            official_sources={"fast-detect-gpt": "baoguangsheng/fast-detect-gpt@971b0520 scripts/fast_detect_gpt.py, scripts/baselines.py",
                              "binoculars": "ahans30/Binoculars@c8ae2f90 binoculars/metrics.py"},
            python=sys.version, torch=torch.__version__, transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0))
json.dump(info, open(f"{OUT}/run_info.json", "w"), ensure_ascii=False, indent=2); print(json.dumps(info, ensure_ascii=False, indent=2))


In [ ]:
%cd /content
drive.flush_and_unmount(); print("Done: results are in DATA_DIR/10_official_equivalence/")
